In [1]:
!pip install transformers torch gradio

In [2]:
from transformers import pipeline
import spacy
import torch
import gradio as gr
import pandas as pd
from sklearn.linear_model import LinearRegression

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
classifier = pipeline('zero-shot-classification', model = 'facebook/bart-large-mnli')

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [5]:
def classify_prompt(prompt):
    labels = ['descriptive_analysis', 'correlation', 'regression', 'others']
    results = classifier(prompt, candidate_labels = labels)
    return results['labels'][0]

In [6]:
nlp = spacy.load('en_core_web_sm')

def extract_variables(prompt):
    doc = nlp(prompt)
    variables = [ent.text for ent in doc.ents if ent.label_ == 'GPE' or ent.label_ == 'ORG']
    if len(variables) == 0:
        return None
    return variables

In [7]:
def load_data(file):
    if not file.name.endswith('.csv'):
        raise ValueError('Please upload a CSV file.')
    return pd.read_csv(file.name)

def get_columns(df):
    return df.columns.tolist()

In [8]:
def descriptive_statistics(df, variables):
    return df[variables].describe()

def correlation(df, variables):
    return df[variables].corr()

def regression(df, dependent_var, independent_vars):
    X = df[independent_vars]
    y = df[dependent_var]
    model = LinearRegression()
    model.fit(X, y)
    return model.coef_, model.intercept_

In [9]:
def analyze_data(file, prompt):
    analysis_type = classify_prompt(prompt)
    variables = extract_variables(prompt)

    if variables is None:
        return 'Please specify the variables for analysis.'

    try:
        df = load_data(file)
    except ValueError as e:
        return str(e)

    columns = get_columns(df)

    if set(variables).issubset(columns):
        if analysis_type == 'descriptive_analysis':
            result = descriptive_statistics(df, variables)
        elif analysis_type == 'correlation':
            result = correlation(df, variables)
        elif analysis_type == 'regression':
            result = regression(df, variables[0], variables[1])
        else:
            result = 'Analysis type not recognized.'
    else:
        return 'Some variables not found in the dataset.'
    
    return result

In [10]:
interface = gr.Interface(fn = analyze_data,
                        inputs=[gr.File(label = 'Upload CSV File'), gr.Textbox(label = 'Analysis Prompt')],
                        outputs='text',
                        title= 'Gen AI Data Analysis Assistant')
interface.launch(debug = True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://482b7d0b24be66195e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


: 